In [1]:
# Expt. No. 10 (manual)
# Title: Design and implement an LSTM model for sentiment analysis

import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.datasets import imdb

vocab_size = 10000
maxlen = 100

(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

X_train_pad = pad_sequences(
    X_train,
    maxlen=maxlen,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test,
    maxlen=maxlen,
    padding='post',
    truncating='post'
)

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

es = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

mc = ModelCheckpoint(
    'best_imdb_lstm_model.keras',
    monitor='val_loss',
    save_best_only=True
)

history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[es, mc]
)

loss, acc = model.evaluate(X_test_pad, y_test)

print(f'Test Loss: {loss:.4f}, Test Accuracy: {acc:.4f}')

word_index = imdb.get_word_index()


def encode_text(text):
    tokens = text.lower().split()

    encoded = [1]

    for word in tokens:
        encoded.append(word_index.get(word, 2))

    return pad_sequences(
        [encoded],
        maxlen=maxlen,
        padding='post',
        truncating='post'
    )


def predict_sentiment(text):
    encoded = encode_text(text)
    prediction = model.predict(encoded, verbose=0)[0][0]

    return f"{'Positive' if prediction > 0.5 else 'Negative'} ({prediction:.2f})"


print(predict_sentiment(
    "This movie was fantastic! I loved every moment of it."
))

print(predict_sentiment(
    "Worst movie ever. Don't waste your time."
))


# OUTPUT:
#
# Test Loss: Depends on training
# Test Accuracy: Depends on training
#
# Positive (prediction score)
# Negative (prediction score)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 54s 165ms/step - accuracy: 0.7179 - loss: 0.5494 - val_accuracy: 0.6314 - val_loss: 0.7393
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 79s 157ms/step - accuracy: 0.8328 - loss: 0.3927 - val_accuracy: 0.8254 - val_loss: 0.4078
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 50s 158ms/step - accuracy: 0.8889 - loss: 0.2967 - val_accuracy: 0.8074 - val_loss: 0.4805
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 50s 161ms/step - accuracy: 0.8819 - loss: 0.3279 - val_accuracy: 0.8012 - val_loss: 0.4577
782/782 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.8079 - loss: 0.4484
Test Loss: 0.4484, Test Accuracy: 0.8079
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Positive (0.96)
Positive (0.96)
